# Regrid population data to match EUs 0.05° x 0.1°

Population count data used here are at a 0.05°x0.05° resolution from Dominik Paprotny (citation to come soon).

Key facts driving the approach:
  - Population lat spacing (0.05) already matches the EU grid's lat spacing
    exactly, with the same cell-center convention -> no lat aggregation
    needed, just alignment/selection.
  - Population lon spacing (0.05) is exactly half the EU grid's lon spacing
    (0.10) -> pairs of adjacent population lon cells sum cleanly into one
    EU lon cell, with centers symmetric around the EU cell center (no
    fractional/partial overlap to worry about).
  - Population COUNT must be summed, not averaged, when downsampling --
    a mean would shrink your totals by not accounting for the
    fact that each output cell now represents 2x the land area.

In [7]:
import os
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [17]:
PM_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "population")

years = [2040, 2060, 2080, 2100]
scenarios = ["H", "HL", "L", "LN", "M", "ML", "VL"]

for scenario in scenarios:
    pop_file = f"Population_count_3min_2040_2100_{scenario}.nc"
    pop_path = os.path.join(POP_DIR, pop_file)
    pop_da = xr.open_dataarray(pop_path)
    for year in years:
        eu_file = f"EU_concentration_{scenario}_{year}.nc"
        eu_path = os.path.join(PM_DIR, eu_file)
        eu = xr.open_dataarray(eu_path)
        pop = pop_da.sel(time=str(year)).squeeze("time", drop=True)

        # Crop to EU bbox with buffer (cheap, avoids processing the globe)
        lat_min, lat_max = float(eu.latitude.min()), float(eu.latitude.max())
        lon_min, lon_max = float(eu.longitude.min()), float(eu.longitude.max())
        buffer = 0.5

        pop_eu = pop.sel(
            lat=slice(lat_min - buffer, lat_max + buffer),
            lon=slice(lon_min - buffer, lon_max + buffer),
        )

        # Align longitude so coarsen() groups the correct cell pairs -----
        # Each EU lon cell (width 0.10) should be the sum of exactly the two
        # population cells whose centers are +/-0.025 from the EU center.
        # Find the population lon index whose center matches the west edge of the
        # first EU cell, so the pairing starts on the right boundary.
        first_eu_lon_west_edge_center = float(eu.longitude.min()) - 0.025
        lon_start_idx = int(np.argmin(np.abs(pop_eu.lon.values - first_eu_lon_west_edge_center)))
        pop_eu = pop_eu.isel(lon=slice(lon_start_idx, lon_start_idx + 2 * eu.longitude.size))

        assert pop_eu.lon.size == 2 * eu.longitude.size, (
            f"Longitude slice came up short ({pop_eu.lon.size} vs "
            f"{2 * eu.longitude.size} needed) -- increase buffer above."
        )

        # Sum pairs of lon cells (conservative aggregation for a count) --
        pop_coarse = pop_eu.coarsen(lon=2, boundary="exact").sum()

        # Sanity check: coarsened centers should land within 1e-6 of EU centers
        pop_coarse = pop_coarse.assign_coords(lon=eu.longitude.values)

        # Align latitude -- should already match exactly, just confirm --
        # and select exactly the EU lat rows (trims the buffer back off)
        lat_diffs = np.abs(pop_coarse.lat.values[:, None] - eu.latitude.values[None, :])
        nearest_lat_diff = lat_diffs.min(axis=0)
        max_mismatch = float(nearest_lat_diff.max())
        print(f"Max lat mismatch after 'exact' alignment check: {max_mismatch:.6f} degrees")
        if max_mismatch > 1e-6:
            print("WARNING: lat grids don't align as expected -- check offsets before trusting output.")

        pop_final = pop_coarse.sel(lat=eu.latitude.values, method="nearest")
        pop_final = pop_final.assign_coords(lat=eu.latitude.values)  # snap exactly
        pop_final = pop_final.rename({"lat": "latitude", "lon": "longitude"})

        # Final checks
        assert pop_final.shape == eu.shape, f"Shape mismatch: {pop_final.shape} vs {eu.shape}"
        print("Output shape:", pop_final.shape, "-- matches EU grid:", eu.shape)

        total_before = float(pop_eu.sum())
        total_after = float(pop_final.sum())
        print(f"Total population before aggregation (EU bbox, incl. buffer): {total_before:,.0f}")
        print(f"Total population after aggregation (trimmed to EU extent):   {total_after:,.0f}")
        print("(These won't match exactly -- the buffer region gets trimmed off when aligning latitude "
              "when snapping to the exact EU lat rows. A huge mismatch would flag a bug; "
              "a modest one from the buffer trim is expected.)")

        pop_final.name = "Population_count_2040"
        out_file = f"Population_count_regridded_{year}_{scenario}.nc"
        out_path = os.path.join(POP_DIR, out_file)
        pop_final.to_netcdf(out_path)
        print(f"Saved: {out_path}")

print("All processing complete.")

Max lat mismatch after 'exact' alignment check: 0.000000 degrees
Output shape: (781, 601) -- matches EU grid: (781, 601)
Total population before aggregation (EU bbox, incl. buffer): 975,846,016
Total population after aggregation (trimmed to EU extent):   955,511,872
(These won't match exactly -- the buffer region gets trimmed off when aligning latitude when snapping to the exact EU lat rows. A huge mismatch would flag a bug; a modest one from the buffer trim is expected.)
Saved: /glade/work/awells/EU_pm/population/Population_count_regridded_2040_H.nc
Max lat mismatch after 'exact' alignment check: 0.000000 degrees
Output shape: (781, 601) -- matches EU grid: (781, 601)
Total population before aggregation (EU bbox, incl. buffer): 1,017,883,968
Total population after aggregation (trimmed to EU extent):   991,953,792
(These won't match exactly -- the buffer region gets trimmed off when aligning latitude when snapping to the exact EU lat rows. A huge mismatch would flag a bug; a modest one